<a href="https://colab.research.google.com/github/Peeyusj/week7_activations_gradients/blob/main/week7_activations_gradients.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt

# Load data
url = "https://raw.githubusercontent.com/Peeyusj/makeMoreIndia/main/indian_names.csv"
df = pd.read_csv(url)
words = df['Name'].str.lower().tolist()
words = [w for w in words if isinstance(w, str) and w.isalpha()]
import random
random.seed(42)
random.shuffle(words)

print(f"Total words: {len(words)}")

Total words: 6466


In [3]:
chars = sorted(set(''.join(words)))
chars = ['.'] + chars
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
print(chars)
print(len(chars))

['.', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
27


In [4]:
C=torch.randn(27,2)
W1=torch.randn(6,100)
b1=torch.randn(100)

In [5]:
block_size=3
X,Y=[], []
for w in words:
 context=[0]*block_size
 for ch in w+'.':
  ix=stoi[ch]
  X.append(context)           # append FIRST
  Y.append(ix)
  context = context[1:] + [ix] # slide LAST

X = torch.tensor(X)
Y = torch.tensor(Y)
print(X.shape)
print(Y.shape)
print(X[:5])
print(Y[:5])

torch.Size([47561, 3])
torch.Size([47561])
tensor([[ 0,  0,  0],
        [ 0,  0, 19],
        [ 0, 19,  8],
        [19,  8,  5],
        [ 8,  5, 11]])
tensor([19,  8,  5, 11,  8])


In [6]:
# Split data
n1 = int(0.8 * len(X))   # 80% mark
n2 = int(0.9 * len(X))   # 90% mark

Xtr, Ytr = X[:n1], Y[:n1]           # training
Xval, Yval = X[n1:n2], Y[n1:n2]     # validation
Xtest, Ytest = X[n2:], Y[n2:]       # test

print(f"Train: {Xtr.shape[0]}, Val: {Xval.shape[0]}, Test: {Xtest.shape[0]}")

Train: 38048, Val: 4756, Test: 4757


In [7]:
# Reinitialize EVERYTHING
C  = torch.randn(27, 5)
W1 = torch.randn(15, 100) * (5/3) / (15 ** 0.5)
b1 = torch.randn(100) * 0
W2 = torch.randn(100, 27) * 0.01
b2 = torch.randn(27) * 0

# BatchNorm parameters
bngain = torch.ones(100)
bnbias = torch.zeros(100)
bnmean_running = torch.zeros(100)
bnstd_running = torch.ones(100)

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
for p in parameters:
    p.requires_grad = True

for i in range(20000):
    ix = torch.randint(0, Xtr.shape[0], (32,))
    emb = C[Xtr[ix]]

    # Linear layer
    preact = emb.view(-1, 15) @ W1 + b1

    # BatchNorm step 1: normalize
    preact_mean = preact.mean(dim=0)
    preact_std = preact.std(dim=0)
    preact_normalized = (preact - preact_mean) / preact_std

    # BatchNorm step 2: scale and shift
    preact_out = bngain * preact_normalized + bnbias

    # BatchNorm step 3: update running stats
    with torch.no_grad():
        bnmean_running = 0.999 * bnmean_running + 0.001 * preact_mean
        bnstd_running = 0.999 * bnstd_running + 0.001 * preact_std

    # Tanh + output
    h = torch.tanh(preact_out)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Ytr[ix])

    for p in parameters:
        p.grad = None
    loss.backward()

    lr = 0.1 if i < 10000 else 0.01
    for p in parameters:
        p.data += -lr * p.grad

    if i % 3000 == 0:
        print(f"Step {i}, Loss: {loss.item():.4f}")

# Evaluate using running stats
emb = C[Xtr]
preact = emb.view(-1, 15) @ W1 + b1
preact_normalized = (preact - bnmean_running) / bnstd_running
preact_out = bngain * preact_normalized + bnbias
h = torch.tanh(preact_out)
logits = h @ W2 + b2
train_loss = F.cross_entropy(logits, Ytr)

emb = C[Xval]
preact = emb.view(-1, 15) @ W1 + b1
preact_normalized = (preact - bnmean_running) / bnstd_running
preact_out = bngain * preact_normalized + bnbias
h = torch.tanh(preact_out)
logits = h @ W2 + b2
val_loss = F.cross_entropy(logits, Yval)

print(f"Train loss: {train_loss.item():.4f}")
print(f"Val loss: {val_loss.item():.4f}")

Step 0, Loss: 3.3088
Step 3000, Loss: 2.4819
Step 6000, Loss: 2.3575
Step 9000, Loss: 2.0337
Step 12000, Loss: 2.0070
Step 15000, Loss: 2.0084
Step 18000, Loss: 2.3032
Train loss: 2.1292
Val loss: 2.1662


In [9]:
for _ in range(15):
    name = []
    context = [0] * block_size

    while True:
        emb = C[torch.tensor([context])]
        preact = emb.view(-1, 15) @ W1 + b1
        # Use running stats for inference
        preact_normalized = (preact - bnmean_running) / bnstd_running
        preact_out = bngain * preact_normalized + bnbias
        h = torch.tanh(preact_out)
        logits = h @ W2 + b2
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1).item()

        if ix == 0:
            break
        name.append(itos[ix])
        context = context[1:] + [ix]

    print(''.join(name))

sum
ceti
simti
shimi
nanjjali
jash
dhalin
pamdal
pampet
miimu
map
praghandeeshweetu
riz
ama
uda
